# 171 Starbucks vs 1,200 Competitors: Moran's I, Ripley's K & Spatial Clustering in Manhattan (Reusable Template)

*A copy-paste spatial analysis framework you can apply to any city × any brand*

**Series context:** This is Theme 2A (spatial clustering) in the Starbucks data science series. See also:
- [Manhattan Cafe Wars](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) — EDA & competitor mapping (Theme 0)
- [Starbucks 10-K NLP](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) — Keyword trends, LDA topics (Theme 1)
- [Starbucks Location Fitness](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) — Demand-supply scoring & backtest (Theme 2B)
- [Data Pipeline](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) — Full reproducibility pipeline (Notebook C)
- [Walk-Distance Analysis](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) — OSMnx walk-distance analysis (Theme 2C)
- [LDA Topic Explorer](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) — Interactive LDA exploration (Theme 1F)

---

**What you'll learn from this notebook:**
1. **Spatial autocorrelation** (Moran's I) — are Starbucks locations randomly scattered or statistically clustered?
2. **LISA hotspot detection** — which Census Tracts are over-served vs under-served?
3. **Point pattern analysis** (Ripley's K) — at what distance scale is clustering strongest?
4. **Transit-based store typology** — how subway ridership patterns reveal different store strategies

**The punchline (spoiler):**
> *Starbucks locations correlate with subway ridership (r = 0.58) but NOT with household income (r = 0.03). They don't chase rich neighborhoods — they chase foot traffic.*

## Section 0 — Setup & Data Loading

We load `stores_enriched_v4.csv` — a pre-built table of 171 Manhattan Starbucks locations enriched with building attributes (MapPLUTO), nearest subway station + ridership (MTA), competitor counts within 250m/500m/1000m, Census Tract demographics (ACS), demand/supply indices, Location Fitness Scores, and station ridership cluster assignments.

All coordinates are WGS 84 (EPSG:4326). Distance calculations use a simple planar approximation valid for Manhattan's scale (error < 0.5%).

> **Reuse this template:** Replace the CSV with your own point data (lat, lon, category) and the same analysis pipeline works for any city.

In [ ]:
# Kaggle environment: install spatial analysis libraries
!pip install -q libpysal esda geopandas folium plotly

import pandas as pd
import numpy as np
import folium
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import cKDTree
from pathlib import Path
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# ----- Data path auto-detect (Kaggle vs local) -----
KAGGLE_PATHS = [
    Path("/kaggle/input/manhattan-cafe-wars"),
    Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
]
LOCAL_DIR = Path("../../data/processed")

DATA_DIR = None
ON_KAGGLE = False
for p in KAGGLE_PATHS:
    if p.exists():
        DATA_DIR = p
        ON_KAGGLE = True
        break
if DATA_DIR is None and LOCAL_DIR.exists():
    DATA_DIR = LOCAL_DIR
if DATA_DIR is None:
    raise FileNotFoundError(
        "Data not found. On Kaggle, add dataset 'shiratoriseto/manhattan-cafe-wars'. "
        "Locally, ensure data/processed/ contains the CSV files."
    )
print(f"Data directory: {DATA_DIR}")

def show_map(m):
    if ON_KAGGLE:
        display(HTML(m._repr_html_()))
    else:
        display(m)

In [ ]:
# ----- Load enriched store data -----
df = pd.read_csv(DATA_DIR / "stores_enriched_v4.csv")
cafes = pd.read_csv(DATA_DIR / "manhattan_cafes_osm.csv")

print(f"Starbucks stores : {len(df):,}")
print(f"All cafés        : {len(cafes):,}")
print(f"Enriched columns : {df.shape[1]}")
print(f"\nColumn groups:")
print(f"  OSM attributes  : osm_id, name, addr_*, lat, lon")
print(f"  Building (PLUTO): pluto_landuse, pluto_numfloors, pluto_retailarea, ...")
print(f"  Transit (MTA)   : mta_station_name, mta_dist_m, mta_avg_daily_ridership")
print(f"  Competition     : n_starbucks_500m, n_dunkin_500m, n_other_cafe_500m, ...")
print(f"  Demographics    : tract_population, tract_median_income, ...")
print(f"  Demand/Supply   : demand_proxy_index, supply_index, location_fitness_score")
print(f"  Station cluster : station_cluster_name")

assert len(df) == 171, f"Expected 171, got {len(df)}"

# Derived column used throughout
df["total_competitors_500m"] = (
    df["n_starbucks_500m"] + df["n_dunkin_500m"] + df["n_other_cafe_500m"]
)

# Distance helper
REF_LAT = 40.75
M_PER_DEG_LAT = 111_320
M_PER_DEG_LON = M_PER_DEG_LAT * np.cos(np.radians(REF_LAT))

def to_meters(lat, lon):
    return lat * M_PER_DEG_LAT, lon * M_PER_DEG_LON

## Section 1 — The Hook: What Drives Starbucks Location?

Before any statistical modeling, one question: **what predicts where Starbucks opens a store?**

The intuitive answer is "wealthy neighborhoods." The data says otherwise.

In [ ]:
# ----- Fig. 1: The 5-second map — size = ridership, color = competitor density -----
fig = px.scatter_map(
    df,
    lat="lat", lon="lon",
    size="mta_avg_daily_ridership",
    color="total_competitors_500m",
    color_continuous_scale="YlOrRd",
    size_max=25, opacity=0.8,
    hover_name="addr_street",
    hover_data={
        "mta_station_name": True,
        "mta_avg_daily_ridership": ":,.0f",
        "total_competitors_500m": True,
        "tract_median_income": ":$,.0f",
        "lat": False, "lon": False,
    },
    labels={
        "total_competitors_500m": "Competitors (500m)",
        "mta_avg_daily_ridership": "Station Ridership",
        "tract_median_income": "Tract Income",
    },
    title="Fig. 1 — Manhattan Starbucks: Size = Nearest Station Ridership, Color = Competitor Density",
    zoom=11.5,
    center={"lat": 40.76, "lon": -73.975},
    height=650,
)
fig.update_layout(
    coloraxis_colorbar_title_text="Competitors<br>(500m)",
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

In [ ]:
# ----- Fig. 2: The punchline — ridership vs income as location predictors -----
r_ridership = df["mta_avg_daily_ridership"].corr(df["total_competitors_500m"])
r_income = df["tract_median_income"].corr(df["total_competitors_500m"])

fig = make_subplots(rows=1, cols=2,
    subplot_titles=[
        f"Subway Ridership vs Competitors (r = {r_ridership:.2f})",
        f"Household Income vs Competitors (r = {r_income:.2f})",
    ])

fig.add_trace(go.Scatter(
    x=df["mta_avg_daily_ridership"], y=df["total_competitors_500m"],
    mode="markers", marker=dict(color="#D62828", opacity=0.6, size=6),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df["tract_median_income"], y=df["total_competitors_500m"],
    mode="markers", marker=dict(color="#2196F3", opacity=0.6, size=6),
), row=1, col=2)

fig.update_xaxes(title_text="Avg Daily Ridership", row=1, col=1)
fig.update_xaxes(title_text="Median Household Income ($)", row=1, col=2)
fig.update_yaxes(title_text="Total Competitors within 500m", row=1, col=1)
fig.update_layout(
    height=400, showlegend=False, template="plotly_white",
    title_text="Fig. 2 — What Predicts Café Density? Foot Traffic Wins.",
)
fig.show()

print(f"Correlation with competitor density:")
print(f"  Subway ridership : r = {r_ridership:.3f}  <- STRONG")
print(f"  Household income : r = {r_income:.3f}  <- essentially ZERO")

**Takeaway (Fig. 1 & 2):** Starbucks — and its competitors — cluster around **foot traffic**, not wealth. The correlation between subway ridership and café density is r = 0.58, while household income shows essentially zero correlation (r = 0.03).

This isn't unique to Starbucks. It reflects a general principle of urban retail: transit nodes generate pedestrian flow, and pedestrian flow generates demand for grab-and-go food & beverage.

> **Template note:** To test this for your city, replace `mta_avg_daily_ridership` with any foot-traffic proxy (transit ridership, pedestrian counts, POI density).

## Section 2 — How Clustered Are They? Global Moran's I

The map *looks* clustered, but is it statistically significant or just random variation? **Moran's I** is the standard test for spatial autocorrelation — it answers: "are nearby areas more similar than expected by chance?"

**Inputs:**
- A value for each spatial unit (Starbucks count per Census Tract)
- A spatial weights matrix defining which Tracts are "neighbors"

**Output:**
- Moran's I: ranges from −1 (perfect dispersion) through 0 (random) to +1 (perfect clustering)
- A p-value for statistical significance

We use **Queen contiguity** — two Census Tracts are neighbors if they share any boundary point (including corners).

> **Template note:** This 10-line recipe works for *any* count variable on *any* polygon layer. Swap in crime counts, restaurant density, or housing prices.

In [ ]:
# ----- Aggregate to Census Tract level -----
import geopandas as gpd
from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local

# Count Starbucks per tract
df["census_tract_id"] = df["census_tract_id"].astype(str)
tract_counts = df.groupby("census_tract_id").size().reset_index(name="starbucks_count")

# Load tract polygons
try:
    tracts_gdf = gpd.read_file(DATA_DIR / "manhattan_tracts_lisa.geojson")
    # Drop pre-computed columns if present — we'll recompute
    drop_cols = [c for c in ["starbucks_count", "lisa_cluster", "lisa_q", "lisa_p",
                              "lisa_sig", "lisa_I", "census_tract_id"] if c in tracts_gdf.columns]
    tracts_gdf = tracts_gdf.drop(columns=drop_cols)
    print(f"Loaded tract polygons: {len(tracts_gdf)} tracts")
except Exception:
    import urllib.request, zipfile, os
    tiger_url = "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_36_tract.zip"
    os.makedirs("/tmp/tiger", exist_ok=True)
    zip_path = "/tmp/tiger/tl_2023_36_tract.zip"
    if not os.path.exists(zip_path):
        print("Downloading TIGER/Line tract boundaries...")
        urllib.request.urlretrieve(tiger_url, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/tmp/tiger")
    tracts_gdf = gpd.read_file("/tmp/tiger/tl_2023_36_tract.shp")
    tracts_gdf = tracts_gdf[tracts_gdf["COUNTYFP"] == "061"].to_crs(epsg=4326)
    print(f"Downloaded TIGER/Line: {len(tracts_gdf)} Manhattan tracts")

# Merge store counts
tracts_gdf = tracts_gdf.merge(tract_counts, left_on="GEOID", right_on="census_tract_id", how="left")
tracts_gdf["starbucks_count"] = tracts_gdf["starbucks_count"].fillna(0).astype(int)

print(f"Tracts with >= 1 Starbucks: {(tracts_gdf['starbucks_count'] > 0).sum()} / {len(tracts_gdf)}")
print(f"Max stores in one tract : {tracts_gdf['starbucks_count'].max()}")

In [ ]:
# ----- Spatial weights + Global Moran's I -----
w = Queen.from_dataframe(tracts_gdf, use_index=False)
if w.islands:
    print(f"Removing {len(w.islands)} island tract(s)")
    tracts_gdf = tracts_gdf.drop(w.islands).reset_index(drop=True)
    w = Queen.from_dataframe(tracts_gdf, use_index=False)

w.transform = "R"
y = tracts_gdf["starbucks_count"].values

mi = Moran(y, w, permutations=9999)

print("=" * 55)
print("GLOBAL MORAN'S I  —  Starbucks Count by Census Tract")
print("=" * 55)
print(f"  Moran's I  : {mi.I:.4f}")
print(f"  Expected   : {mi.EI:.4f}  (under spatial randomness)")
print(f"  z-score    : {mi.z_sim:.2f}")
print(f"  p-value    : {mi.p_sim:.6f}")
print()
if mi.I > 0 and mi.p_sim < 0.05:
    print("  Result: SIGNIFICANT positive spatial autocorrelation.")
    print("  Starbucks stores are spatially CLUSTERED, not randomly placed.")

**Moran's I = 0.36, p < 0.001** — Starbucks locations are far more clustered than random chance would produce. A z-score of 11.3 means the null hypothesis of spatial randomness can be rejected with overwhelming confidence.

But Global Moran's I only tells us that clustering exists *somewhere* — it doesn't say *where*. For that, we need **Local Moran's I (LISA)**.

## Section 3 — Where Are the Hotspots? LISA Cluster Map

**LISA (Local Indicators of Spatial Association)** decomposes Global Moran's I into contributions from each Census Tract:

| Cluster | Meaning | Color |
|---------|---------|-------|
| **High-High** | High store count surrounded by high — **hotspot** | Red |
| **Low-Low** | Low count surrounded by low — **coldspot** | Blue |
| **High-Low** | High surrounded by low — **spatial outlier** | Orange |
| **Low-High** | Low surrounded by high — **gap in a hot area** | Light blue |

Only statistically significant results (p < 0.05) are colored.

In [ ]:
# ----- LISA computation -----
lisa = Moran_Local(y, w, permutations=9999)

sig = lisa.p_sim < 0.05
labels_map = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}

tracts_gdf["lisa_cluster"] = "Not Significant"
for q, label in labels_map.items():
    mask = (lisa.q == q) & sig
    tracts_gdf.loc[mask, "lisa_cluster"] = label

print("LISA Cluster Summary:")
print(tracts_gdf["lisa_cluster"].value_counts().to_string())

In [ ]:
# ----- Fig. 3: LISA choropleth map -----
cluster_colors = {
    "High-High": "#d7191c",
    "Low-Low": "#2c7bb6",
    "High-Low": "#fdae61",
    "Low-High": "#abd9e9",
    "Not Significant": "#eeeeee",
}

m = folium.Map(location=[40.78, -73.965], zoom_start=12, tiles="CartoDB positron")

folium.GeoJson(
    tracts_gdf,
    style_function=lambda feat: {
        "fillColor": cluster_colors.get(feat["properties"]["lisa_cluster"], "#eee"),
        "color": "#333",
        "weight": 0.5,
        "fillOpacity": 0.7 if feat["properties"]["lisa_cluster"] != "Not Significant" else 0.15,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NAMELSAD", "starbucks_count", "lisa_cluster"],
        aliases=["Tract", "Starbucks #", "Cluster"],
    ),
).add_to(m)

legend = f'''
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px 16px;border:1px solid #999;
     border-radius:4px;font-size:13px;line-height:1.8;">
<b>Fig. 3 — LISA Clusters (p < 0.05)</b><br>
<span style="color:#d7191c">&#9632;</span> High-High (Hotspot)<br>
<span style="color:#2c7bb6">&#9632;</span> Low-Low (Coldspot)<br>
<span style="color:#fdae61">&#9632;</span> High-Low (Outlier)<br>
<span style="color:#abd9e9">&#9632;</span> Low-High (Gap in hot area)<br>
<span style="color:#eeeeee">&#9632;</span> Not Significant<br>
<hr style="margin:4px 0"><b>Global Moran\'s I = {mi.I:.3f}</b> (p < 0.001)
</div>
'''
m.get_root().html.add_child(folium.Element(legend))
show_map(m)

In [ ]:
# ----- Geographic breakdown -----
tracts_gdf["centroid_lat"] = tracts_gdf.geometry.centroid.y

def area_label(lat):
    if lat >= 40.82: return "Upper Manhattan (Harlem+)"
    if lat >= 40.795: return "Upper West/East Side"
    if lat >= 40.765: return "Midtown"
    if lat >= 40.74: return "Chelsea / Murray Hill"
    if lat >= 40.72: return "Greenwich / East Village"
    return "Lower Manhattan / FiDi"

tracts_gdf["area"] = tracts_gdf["centroid_lat"].apply(area_label)

print("HIGH-HIGH (Hotspot) tracts by area:")
hh = tracts_gdf[tracts_gdf["lisa_cluster"] == "High-High"]
print(hh.groupby("area")["starbucks_count"].agg(["count", "sum"]).rename(
    columns={"count": "tracts", "sum": "stores"}).sort_values("tracts", ascending=False).to_string())

print("\nLOW-HIGH (Gap in hot area) — potential opportunity zones:")
lh = tracts_gdf[tracts_gdf["lisa_cluster"] == "Low-High"]
if len(lh) > 0:
    print(lh.groupby("area")["starbucks_count"].agg(["count", "sum"]).rename(
        columns={"count": "tracts", "sum": "stores"}).to_string())
else:
    print("  None detected")

**Key findings (Fig. 3):**

1. **30 hotspot tracts** (High-High) — 77% concentrated in Chelsea / Murray Hill. This is the core of Starbucks' dominant territory in Manhattan.
2. **5 coldspot tracts** (Low-Low) — Upper Manhattan and Upper West/East Side, where overall commercial density is lower.
3. **9 "Low-High" tracts** — the most interesting result. These sit *inside* hot zones but have few or no Starbucks. They represent either genuine coverage gaps or locations where demand doesn't justify a store despite nearby density.

The LISA map transforms a vague impression ("Starbucks clusters in Midtown") into a precise, tract-level classification that is testable and actionable.

> **Template note:** This exact pipeline — `Queen.from_dataframe()` → `Moran()` → `Moran_Local()` → folium choropleth — works for any count variable on any polygon layer.

## Section 4 — Tighter Than the Competition: Nearest Neighbor Distances & Ripley's K

Moran's I proved clustering exists at the Census Tract level. Now we zoom in to the **point level** — measuring the actual distance between individual stores.

Two questions:
1. **How close does Starbucks place its own stores to each other?** (self-cannibalization)
2. **Is Starbucks spacing tighter or looser than Dunkin'?** (dominant strategy signal)

In [ ]:
# ----- Nearest-neighbor distances: Starbucks vs Dunkin' -----
sbux_y, sbux_x = to_meters(df["lat"].values, df["lon"].values)
sbux_coords = np.column_stack([sbux_x, sbux_y])

# Starbucks → nearest Starbucks (k=2: skip self)
tree_sbux = cKDTree(sbux_coords)
dist_sbux, _ = tree_sbux.query(sbux_coords, k=2)
nn_sbux = dist_sbux[:, 1]

# Dunkin' → nearest Dunkin'
dunkin = cafes[cafes["brand_category"] == "dunkin"]
dunk_y, dunk_x = to_meters(dunkin["lat"].values, dunkin["lon"].values)
dunk_coords = np.column_stack([dunk_x, dunk_y])
tree_dunk = cKDTree(dunk_coords)
dist_dunk, _ = tree_dunk.query(dunk_coords, k=2)
nn_dunk = dist_dunk[:, 1]

# Summary
ratio = np.median(nn_dunk) / np.median(nn_sbux)
print("Nearest Same-Brand Neighbor:")
print(f"  Starbucks (n={len(nn_sbux)}): median = {np.median(nn_sbux):.0f}m, mean = {np.mean(nn_sbux):.0f}m")
print(f"  Dunkin'   (n={len(nn_dunk)}): median = {np.median(nn_dunk):.0f}m, mean = {np.mean(nn_dunk):.0f}m")
print(f"\n  Ratio: Dunkin' / Starbucks = {ratio:.2f}x")
print(f"  -> Starbucks packs its stores {ratio:.1f}x tighter than Dunkin' does.")
print(f"\n  Starbucks within 200m of another Starbucks: {(nn_sbux < 200).sum()} ({(nn_sbux < 200).mean():.0%})")

In [ ]:
# ----- Fig. 4: Overlay histogram — nearest same-brand distances -----
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=nn_sbux, nbinsx=40, name=f"Starbucks→Starbucks (n={len(nn_sbux)})",
    marker_color="#D62828", opacity=0.7,
))
fig.add_trace(go.Histogram(
    x=nn_dunk, nbinsx=30, name=f"Dunkin'→Dunkin' (n={len(nn_dunk)})",
    marker_color="#FF8C00", opacity=0.7,
))

# Median lines
fig.add_vline(x=np.median(nn_sbux), line_dash="dash", line_color="#D62828",
              annotation_text=f"Starbucks median: {np.median(nn_sbux):.0f}m", annotation_position="top right")
fig.add_vline(x=np.median(nn_dunk), line_dash="dash", line_color="#FF8C00",
              annotation_text=f"Dunkin' median: {np.median(nn_dunk):.0f}m", annotation_position="top left")

fig.update_layout(
    title="Fig. 4 — Nearest Same-Brand Distance: Starbucks Packs 1.4x Tighter Than Dunkin'",
    xaxis_title="Distance to nearest same-brand store (m)",
    yaxis_title="Count",
    barmode="overlay", template="plotly_white",
    height=420, width=900,
    legend=dict(x=0.55, y=0.95),
)
fig.show()

In [ ]:
# ----- Fig. 5: Ripley's L function — clustering intensity by distance -----
def ripley_k(coords, n, area, radii):
    """Compute Ripley's K function for a set of point coordinates."""
    tree = cKDTree(coords)
    K = np.zeros(len(radii))
    for i, r in enumerate(radii):
        counts = tree.query_ball_point(coords, r)
        total = sum(len(c) - 1 for c in counts)
        K[i] = area / (n * (n - 1)) * total
    return K

# Study area bounding box (with margin to reduce edge effects)
margin = 200
all_x = np.concatenate([sbux_x, dunk_x])
all_y = np.concatenate([sbux_y, dunk_y])
x_min, x_max = all_x.min() - margin, all_x.max() + margin
y_min, y_max = all_y.min() - margin, all_y.max() + margin
area = (x_max - x_min) * (y_max - y_min)

radii = np.arange(50, 2050, 50)

# Observed L functions (Besag's L = sqrt(K/pi) - d)
K_sbux = ripley_k(sbux_coords, len(sbux_coords), area, radii)
L_sbux = np.sqrt(K_sbux / np.pi) - radii

K_dunk = ripley_k(dunk_coords, len(dunk_coords), area, radii)
L_dunk = np.sqrt(K_dunk / np.pi) - radii

# CSR envelope (99 Monte Carlo simulations of complete spatial randomness)
np.random.seed(42)
n_sims = 99
L_sims = np.zeros((n_sims, len(radii)))
for s in range(n_sims):
    rand_coords = np.column_stack([
        np.random.uniform(x_min, x_max, len(sbux_coords)),
        np.random.uniform(y_min, y_max, len(sbux_coords)),
    ])
    K_rand = ripley_k(rand_coords, len(sbux_coords), area, radii)
    L_sims[s] = np.sqrt(K_rand / np.pi) - radii

L_upper = np.percentile(L_sims, 97.5, axis=0)
L_lower = np.percentile(L_sims, 2.5, axis=0)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=radii, y=L_upper, mode="lines",
    line=dict(dash="dash", color="grey", width=1), name="CSR envelope (95%)"))
fig.add_trace(go.Scatter(x=radii, y=L_lower, mode="lines",
    line=dict(dash="dash", color="grey", width=1), fill="tonexty",
    fillcolor="rgba(200,200,200,0.2)", showlegend=False))
fig.add_hline(y=0, line_dash="dot", line_color="black", line_width=0.5)
fig.add_trace(go.Scatter(x=radii, y=L_sbux, mode="lines",
    line=dict(color="#D62828", width=2.5), name=f"Starbucks (n={len(sbux_coords)})"))
fig.add_trace(go.Scatter(x=radii, y=L_dunk, mode="lines",
    line=dict(color="#FF8C00", width=2.5), name=f"Dunkin' (n={len(dunk_coords)})"))

fig.update_layout(
    title="Fig. 5 — Besag's L Function: Starbucks Clusters More Intensely Than Dunkin' at Every Distance",
    xaxis_title="Distance d (meters)", yaxis_title="L(d) − d  (> 0 = clustered beyond random)",
    height=450, width=900, template="plotly_white",
    legend=dict(x=0.55, y=0.95),
)
fig.show()

print(f"L(d) at key distances (positive = more clustered than random):")
print(f"  {'Distance':>8s}  {'Starbucks':>10s}  {'Dunkin':>10s}  {'CSR upper':>10s}")
for d in [200, 500, 1000]:
    idx = np.searchsorted(radii, d)
    print(f"  {d:>7d}m  {L_sbux[idx]:>10.0f}  {L_dunk[idx]:>10.0f}  {L_upper[idx]:>10.0f}")

**Key findings (Fig. 4 & 5):**

- **Starbucks median self-spacing: 208m** — nearly half of all stores have another Starbucks within 200m. This is the quantitative fingerprint of a dominant location strategy.
- **Dunkin' median self-spacing: 294m** — Starbucks packs **1.4x tighter** than its main competitor.
- **Ripley's L function:** Starbucks exceeds the random baseline (CSR envelope) at *every* distance from 50m to 2km, and consistently outpaces Dunkin'. The clustering is not just local — it is systematic across all spatial scales.

> **Template note:** The `ripley_k()` function above accepts any set of point coordinates. Feed in your own store locations to measure clustering intensity at any scale.

## Section 5 — Rush Hour Reveals Store Strategy: MTA Hourly Clustering

Sections 2–4 proved that Starbucks locations are clustered. But **not all clusters serve the same demand**. A station in the Financial District sees a morning rush of commuters grabbing coffee before work; a station near Central Park sees steady midday tourist traffic.

We use K-means (k=4) on hourly ridership profiles to classify every Manhattan subway station by **when** its passengers arrive — then cross-reference with nearby Starbucks store density and building attributes.

In [ ]:
# ----- Fig. 6: MTA hourly ridership profiles by station cluster -----
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Load pre-computed clusters (K-means on hourly profiles, k=4)
CLUSTER_PATH = DATA_DIR / "mta_station_clusters.csv"
if not CLUSTER_PATH.exists():
    CLUSTER_PATH = Path("../../data/interim/mta_station_clusters.csv")

if CLUSTER_PATH.exists():
    station_clusters = pd.read_csv(CLUSTER_PATH)
    station_clusters["station_complex_id"] = station_clusters["station_complex_id"].astype(str)
    
    # Load hourly ridership to plot profiles
    MTA_HOURLY_PATH = DATA_DIR / "mta_hourly_manhattan_q4_2024.csv"
    if not MTA_HOURLY_PATH.exists():
        MTA_HOURLY_PATH = Path("../../data/raw/mta_hourly_manhattan_q4_2024.csv")
    
    hourly = pd.read_csv(MTA_HOURLY_PATH)
    hourly["station_complex_id"] = hourly["station_complex_id"].astype(str)
    
    # Merge clusters
    hourly_c = hourly.merge(station_clusters, on="station_complex_id")
    
    # Compute mean hourly profile per cluster (normalized to % of daily total)
    profiles = hourly_c.groupby(["cluster_name", "hour"])["total_ridership"].mean().reset_index()
    daily_totals = profiles.groupby("cluster_name")["total_ridership"].transform("sum")
    profiles["pct"] = profiles["total_ridership"] / daily_totals * 100
    
    # Plot
    cluster_colors = {
        "Morning Peak (Residential)": "#2a9d8f",
        "Balanced (Transit Hub)": "#457b9d",
        "Midday-Heavy (Tourism/Shopping)": "#f4a261",
        "Evening Peak (Office District)": "#e63946",
    }
    
    fig = go.Figure()
    for cname in cluster_colors:
        mask = profiles["cluster_name"] == cname
        fig.add_trace(go.Scatter(
            x=profiles.loc[mask, "hour"],
            y=profiles.loc[mask, "pct"],
            mode="lines+markers",
            name=f"{cname} ({len(station_clusters[station_clusters['cluster_name']==cname])})",
            line=dict(color=cluster_colors[cname], width=2.5),
            marker=dict(size=4),
        ))
    
    fig.update_layout(
        title="Fig. 6 — MTA Hourly Ridership Profiles by Station Cluster (K=4)",
        xaxis_title="Hour of Day",
        yaxis_title="% of Daily Ridership",
        xaxis=dict(tickmode="linear", dtick=2),
        template="plotly_white",
        height=400,
        legend=dict(x=0.55, y=0.98),
    )
    fig.show()
    
    print(f"\nStation cluster distribution:")
    print(station_clusters["cluster_name"].value_counts().to_string())
else:
    print("⚠ mta_station_clusters.csv not found. Skipping hourly clustering.")

In [ ]:
# ----- Fig. 7: Station cluster × Starbucks store attributes cross-tabulation -----
if CLUSTER_PATH.exists():
    # Join station clusters to nearest Starbucks stores
    # Each Starbucks has a nearest_mta_station_complex_id from the enrichment pipeline
    mta_summary_path = DATA_DIR / "manhattan_mta_ridership_summary.csv"
    if not mta_summary_path.exists():
        mta_summary_path = Path("../../data/processed/manhattan_mta_ridership_summary.csv")
    
    mta_summary = pd.read_csv(mta_summary_path)
    mta_summary["station_complex_id"] = mta_summary["station_complex_id"].astype(str)
    
    # Map each station to its cluster
    station_to_cluster = dict(zip(
        station_clusters["station_complex_id"],
        station_clusters["cluster_name"]
    ))
    
    # For each Starbucks, find nearest station and assign cluster
    REF_LAT = 40.75
    M_PER_DEG_LAT = 111_320
    M_PER_DEG_LON = M_PER_DEG_LAT * np.cos(np.radians(REF_LAT))
    
    sbux_xy = np.column_stack([df["lon"] * M_PER_DEG_LON, df["lat"] * M_PER_DEG_LAT])
    mta_xy = np.column_stack([mta_summary["lon"] * M_PER_DEG_LON, mta_summary["lat"] * M_PER_DEG_LAT])
    
    tree = cKDTree(mta_xy)
    dist, idx = tree.query(sbux_xy, k=1)
    
    df["nearest_station_cluster"] = [
        station_to_cluster.get(str(mta_summary.iloc[i]["station_complex_id"]), "Unknown")
        for i in idx
    ]
    df["nearest_station_dist_m"] = dist
    
    # Cross-tabulation: cluster × building/store attributes
    # Nearest same-brand distance (Starbucks-to-Starbucks)
    sbux_tree = cKDTree(sbux_xy)
    dd, ii = sbux_tree.query(sbux_xy, k=2)  # k=2 because k=1 is self
    df["nearest_sbux_dist_m"] = dd[:, 1]
    
    # Aggregate by station cluster
    cluster_order = [
        "Evening Peak (Office District)",
        "Midday-Heavy (Tourism/Shopping)", 
        "Balanced (Transit Hub)",
        "Morning Peak (Residential)",
    ]
    
    agg_data = []
    for cname in cluster_order:
        subset = df[df["nearest_station_cluster"] == cname]
        if len(subset) == 0:
            continue
        agg_data.append({
            "Station Cluster": cname.split(" (")[0],
            "Type": cname.split("(")[1].rstrip(")") if "(" in cname else "",
            "Stores": len(subset),
            "Median Starbucks Spacing (m)": f"{subset['nearest_sbux_dist_m'].median():.0f}",
            "Competitors (500m)": f"{subset['total_competitors_500m'].median():.0f}" if "total_competitors_500m" in subset.columns else "N/A",
            "Avg Building Floors": f"{subset['pluto_numfloors'].median():.0f}" if "pluto_numfloors" in subset.columns else "N/A",
        })
    
    cross_tab = pd.DataFrame(agg_data)
    display(HTML(
        "<h4>Fig. 7 — Station Cluster × Starbucks Store Attributes</h4>" +
        cross_tab.to_html(index=False)
    ))
    
    # The key insight
    if len(agg_data) >= 2:
        office = df[df["nearest_station_cluster"].str.contains("Office")]
        residential = df[df["nearest_station_cluster"].str.contains("Residential")]
        if len(office) > 0 and len(residential) > 0:
            office_spacing = office["nearest_sbux_dist_m"].median()
            res_spacing = residential["nearest_sbux_dist_m"].median()
            ratio = res_spacing / office_spacing
            print(f"\n★ Key insight: Office District Starbucks spacing = {office_spacing:.0f}m")
            print(f"  Residential spacing = {res_spacing:.0f}m ({ratio:.1f}x wider)")
            print(f"  → Office areas pack stores {ratio:.1f}x tighter for grab-and-go commuter demand")

**Key findings (Fig. 6 & 7):**

- **4 distinct station types** emerge from hourly ridership patterns alone — no geographic labels were given to the algorithm
- **Office District stations** show sharp AM/PM peaks → Starbucks responds with **tight spacing (~184m)** for grab-and-go commuter demand
- **Residential stations** show gentler morning peaks → spacing is **~462m (2.5× wider)**, matching lower walk-in density
- **Tourism/Shopping stations** peak midday → moderate spacing with higher competitor counts (tourist areas attract all brands)
- This confirms that Starbucks adapts its **spatial density to match demand temporality**, not just total ridership volume

## Section 6 — Reuse This Template: 3 Steps for Any City × Any Brand

Everything above works for Starbucks in Manhattan, but the real value is the **reusable pipeline**. Here's how to adapt it to your own analysis in three steps.

In [ ]:
# =====================================================
# STEP 1: Prepare your point data
# =====================================================
# You need a CSV with: lat, lon, brand (or category)
# Example: coffee shops in London, pharmacies in Tokyo, etc.
#
# my_stores = pd.read_csv("my_stores.csv")  # must have lat, lon columns
# my_competitors = pd.read_csv("my_competitors.csv")

# =====================================================
# STEP 2: Compute spatial metrics (copy-paste ready)
# =====================================================
def compute_spatial_metrics(stores_df, competitors_df, polygon_gdf,
                            count_col="store_count", radii=[250, 500, 1000]):
    """
    Generic spatial analysis pipeline.
    
    Parameters
    ----------
    stores_df : DataFrame with 'lat', 'lon' columns (your brand)
    competitors_df : DataFrame with 'lat', 'lon' columns (competitors)
    polygon_gdf : GeoDataFrame of area polygons (Census Tracts, wards, etc.)
    count_col : name for the store count column
    radii : list of distances (meters) for competitor density
    
    Returns
    -------
    dict with 'morans_i', 'lisa_gdf', 'nn_distances', 'ripley_L'
    """
    from scipy.spatial import cKDTree
    from libpysal.weights import Queen
    from esda.moran import Moran, Moran_Local
    
    REF_LAT = stores_df["lat"].mean()
    M_LAT = 111_320
    M_LON = M_LAT * np.cos(np.radians(REF_LAT))
    
    # Point coordinates in meters
    s_y, s_x = stores_df["lat"].values * M_LAT, stores_df["lon"].values * M_LON
    s_coords = np.column_stack([s_x, s_y])
    
    # --- Nearest neighbor ---
    tree_s = cKDTree(s_coords)
    dd, _ = tree_s.query(s_coords, k=2)
    nn_self = dd[:, 1]
    
    # --- Moran's I ---
    stores_gdf = gpd.GeoDataFrame(
        stores_df, geometry=gpd.points_from_xy(stores_df["lon"], stores_df["lat"]),
        crs="EPSG:4326")
    joined = gpd.sjoin(stores_gdf, polygon_gdf[["geometry"]], how="left", predicate="within")
    counts = joined.groupby("index_right").size()
    polygon_gdf[count_col] = 0
    polygon_gdf.loc[counts.index, count_col] = counts.values
    
    w = Queen.from_dataframe(polygon_gdf, use_index=False)
    if w.islands:
        polygon_gdf = polygon_gdf.drop(w.islands).reset_index(drop=True)
        w = Queen.from_dataframe(polygon_gdf, use_index=False)
    w.transform = "R"
    y = polygon_gdf[count_col].values
    
    mi = Moran(y, w, permutations=999)
    lisa = Moran_Local(y, w, permutations=999)
    
    return {
        "morans_i": {"I": mi.I, "p": mi.p_sim, "z": mi.z_sim},
        "nn_median": np.median(nn_self),
        "nn_distances": nn_self,
        "polygon_gdf": polygon_gdf,
        "lisa": lisa,
    }

# =====================================================
# STEP 3: Call it
# =====================================================
# results = compute_spatial_metrics(my_stores, my_competitors, my_polygons)
# print(f"Moran's I = {results['morans_i']['I']:.3f}, p = {results['morans_i']['p']:.4f}")
# print(f"Nearest-neighbor median = {results['nn_median']:.0f}m")

print("Template function defined. See docstring for usage.")
print("Replace stores_df, competitors_df, and polygon_gdf with your own data.")

The `compute_spatial_metrics()` function encapsulates the entire pipeline from Sections 2–4:
- Nearest-neighbor distance computation
- Polygon-level aggregation + Global Moran's I
- LISA cluster assignment

**To adapt for your own analysis:**
- `stores_df`: any DataFrame with `lat`, `lon` (your brand's locations)
- `competitors_df`: any DataFrame with `lat`, `lon` (competitor locations)
- `polygon_gdf`: any GeoDataFrame of area polygons (Census Tracts, postal codes, wards)

The math is identical regardless of city, brand, or polygon system.

## Section 7 — Limitations & What's Next

### What this analysis does NOT tell you

| Limitation | Impact | Mitigation |
|---|---|---|
| **OSM coverage ~85%** | ~30 Starbucks stores may be missing from OSM | Results represent a lower bound on clustering intensity |
| **No sales data** | We measure *location* patterns, not *performance* | Subway ridership serves as a demand proxy, not a revenue proxy |
| **Census Tract granularity** | Tracts average ~0.1 km² — too coarse for block-level decisions | Point-level analysis (Section 4) compensates |
| **Snapshot, not time series** | All data reflects 2024 Q4; store openings/closings are not tracked | 10-K filings (Theme 1) provide the historical narrative |
| **Correlation ≠ causation** | "Ridership predicts density" does not mean ridership *causes* store placement | Establishing causality would require a natural experiment or IV design |

### The Series

| Notebook | Topic | Link |
|----------|-------|------|
| **Theme 0** — Manhattan Cafe Wars | EDA, competitor mapping, OSMnx demo | [Manhattan Cafe Wars](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| **Theme 1** — 10-K NLP Analysis | 30 years of keyword trends, LDA topics, NLP × store count | [Starbucks 10-K NLP](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| **Theme 2A** — Spatial Clustering | Moran's I, LISA, Ripley's K | You are here |
| **Theme 2B** — Location Fitness | Demand-supply gap scoring, sensitivity analysis, backtest | [Starbucks Location Fitness](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |

**Notebook B** moves from *description* to *prediction*: which Census Tracts are over-served vs under-served? Which locations should the 172nd Starbucks target?

---

### Data Sources & Licenses

| Source | License | Usage |
|---|---|---|
| OpenStreetMap (store locations) | ODbL 1.0 | Café locations, building footprints |
| MapPLUTO (NYC Dept. of City Planning) | NYC Open Data Terms | Building attributes |
| MTA Subway Hourly Ridership | NY Open Data Terms | Station ridership |
| US Census ACS 2022 5-Year | Public domain | Tract-level demographics |
| TIGER/Line Shapefiles | Public domain | Census Tract boundaries |

---

*Built with pandas · plotly · folium · scipy · libpysal · esda · geopandas*